In [ ]:
!pip -q install --force-reinstall --no-deps 'torch==2.4.1' 'torchvision==0.19.1' 'torchaudio==2.4.1' --index-url https://download.pytorch.org/whl/cu121
!pip -q install --upgrade 'transformers==4.51.3' accelerate 'bitsandbytes==0.43.3'

In [ ]:
import os, subprocess, sys

REPO = "/kaggle/working/lawforge"
if not os.path.isdir(REPO):
    subprocess.check_call(
        ["git", "clone", "--depth", "1", "https://github.com/PAMF2/lawforge.git", REPO]
    )
sys.path.insert(0, REPO)

In [ ]:
import os, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL = os.environ.get("LAWFORGE_LLM_MODEL", "Qwen/Qwen2.5-14B-Instruct")
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL, quantization_config=bnb, device_map="cuda", trust_remote_code=True
)
model.eval()
print(f"loaded {MODEL} mem={torch.cuda.memory_allocated() / 1e9:.2f}GB")

In [ ]:
import json
from pathlib import Path

INPUTS = Path(f"{REPO}/kaggle/llm_classify_v3/inputs")
rows = []
for s in ["hard2_test", "hard3_test"]:
    for line in open(INPUTS / f"{s}.jsonl"):
        r = json.loads(line)
        r["_split"] = s
        rows.append(r)
print(f"rows: {len(rows)}")

In [ ]:
import time, json, re
from pathlib import Path

SYSTEM = (
    "You decide magma equation implications. Output format MUST be: at most 3 short "
    "sentences of reasoning, THEN on the very last line emit exactly one of:\n"
    "ANSWER: TRUE\n"
    "ANSWER: FALSE\n"
    "Do not output anything after that line. Never omit the ANSWER line."
)


@torch.inference_mode()
def classify(h, g):
    user = (
        f"h (universal): {h}\n"
        f"g (universal): {g}\n"
        f"Does h imply g for ALL magmas? Reason briefly (\u22643 sentences) then ANSWER: TRUE or ANSWER: FALSE."
    )
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=600,
        do_sample=False,
        pad_token_id=tok.pad_token_id,
        eos_token_id=tok.eos_token_id,
    )
    txt = tok.decode(out[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True)
    upp = txt.upper()
    m = re.findall(r"ANSWER:?\s*(TRUE|FALSE)", upp)
    if m:
        return m[-1].lower(), txt
    # fallback: look at last 200 chars for words
    tail = upp[-300:]
    has_t = bool(re.search(r"\bTRUE\b", tail))
    has_f = bool(re.search(r"\bFALSE\b", tail))
    if has_t and not has_f:
        return "true", txt
    if has_f and not has_t:
        return "false", txt
    return "unknown", txt


OUT = Path("/kaggle/working/llm_preds_v4.jsonl")
t0 = time.time()
correct = 0
stats = {"tp_t": 0, "fp_t": 0, "tp_f": 0, "fp_f": 0, "unk": 0}
with OUT.open("w") as f:
    for i, r in enumerate(rows):
        pred, raw = classify(r["hypothesis"], r["goal"])
        label = r["label"]
        if pred == "true":
            stats["tp_t" if label == "true" else "fp_t"] += 1
        elif pred == "false":
            stats["tp_f" if label == "false" else "fp_f"] += 1
        else:
            stats["unk"] += 1
        if pred == label:
            correct += 1
        f.write(
            json.dumps(
                {
                    "id": r["id"],
                    "split": r["_split"],
                    "label": label,
                    "pred": pred,
                    "raw": raw[-500:],
                }
            )
            + "\n"
        )
        f.flush()
        if (i + 1) % 20 == 0:
            print(
                f"[{i + 1}/{len(rows)}] correct={correct} stats={stats} t={time.time() - t0:.0f}s",
                flush=True,
            )
print(f"=== FINAL ===", flush=True)
print(f"accuracy={correct}/{len(rows)} = {correct / len(rows) * 100:.1f}%", flush=True)
print(f"stats={stats} time={time.time() - t0:.0f}s", flush=True)